# Substorm Onset Observations by IMAGE-FUV — Implementation / 구현

**Paper**: Frey, H. U., Mende, S. B., Angelopoulos, V., and Donovan, E. F. (2004), Substorm onset observations by IMAGE-FUV, *J. Geophys. Res.*, **109**, A10304, doi:10.1029/2004JA010607.

**EN.** This notebook reproduces three core analyses from the paper, using a synthetic IMAGE-FUV-like onset catalog generated from the statistical properties reported by the authors:
1. The substorm onset detection algorithm (local brightening + 20-min azimuthal expansion + 30-min separation).
2. The MLT histogram of onsets, reproducing the 23:00 ± 01:21 distribution.
3. Magnetic-latitude statistics and the IMF Bz / dynamic-pressure dependences.

**KR.** 이 노트북은 논문에 보고된 통계적 특성에서 생성한 IMAGE-FUV 유사 onset 카탈로그를 사용하여 논문의 세 가지 핵심 분석을 재현한다:
1. Substorm onset 검출 알고리즘 (국지적 밝아짐 + 20분 방위각 확장 + 30분 분리).
2. onset의 MLT 히스토그램으로 23:00 ± 01:21 분포 재현.
3. 자기위도 통계와 IMF Bz / 동압력 의존성.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass
from typing import List, Tuple

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11
rng = np.random.default_rng(seed=20040401)

## Part 1: Onset Detection Algorithm / onset 검출 알고리즘

**EN.** The Frey et al. (2004) onset detection rule has three criteria applied to a time series of FUV images:
1. **Local brightening** — a pixel intensity rises above a threshold above the running background.
2. **Azimuthal expansion ≥ 20 min** — the bright region must spread in MLT for at least 20 minutes.
3. **30-min separation** — events within 30 minutes of a previously accepted onset are rejected.

Below we implement this algorithm on synthetic 1-D MLT-vs-time WIC-like data.

**KR.** Frey et al. (2004)의 onset 검출 규칙은 FUV 영상 시계열에 세 가지 기준을 적용한다:
1. **국지적 밝아짐** — 픽셀 강도가 이동 배경보다 임계값 이상으로 상승.
2. **방위각 확장 ≥ 20분** — 밝은 영역이 MLT 방향으로 20분 이상 확산.
3. **30분 분리** — 이전 onset 후 30분 이내의 사건은 거부.

아래에서는 합성 1차원 MLT-시간 WIC 유사 데이터에 이 알고리즘을 구현한다.

In [ ]:
@dataclass
class OnsetEvent:
    """Container for one detected substorm onset.

    Attributes:
        time_min: Onset time in minutes from start of run.
        mlt_hr: Magnetic local time of the brightest pixel (hours, 0-24).
        mlat_deg: AACGM magnetic latitude of the brightest pixel (deg).
        peak_count: Brightness in instrument counts at onset.
    """
    time_min: float
    mlt_hr: float
    mlat_deg: float
    peak_count: float


def detect_onsets(images: np.ndarray,
                  mlt_grid: np.ndarray,
                  mlat_grid: np.ndarray,
                  cadence_min: float = 2.0,
                  bg_window_min: float = 30.0,
                  n_sigma: float = 3.0,
                  min_expansion_min: float = 20.0,
                  min_expansion_mlt: float = 0.5,
                  min_separation_min: float = 30.0) -> List[OnsetEvent]:
    """Detect substorm onsets from a stack of FUV-like images.

    Implements the three-rule criterion of Frey et al. (2004).

    Args:
        images: Image stack of shape (n_time, n_mlat, n_mlt) in counts.
        mlt_grid: 1-D array of MLT bin centers (hours).
        mlat_grid: 1-D array of magnetic latitude bin centers (deg).
        cadence_min: Image cadence in minutes (IMAGE-FUV: 2 min).
        bg_window_min: Length of running-mean background window (min).
        n_sigma: Threshold multiplier above the background standard deviation.
        min_expansion_min: Minimum azimuthal-expansion duration to accept (min).
        min_expansion_mlt: Minimum MLT extent that the bright region must reach (hours).
        min_separation_min: Minimum gap between two consecutive accepted onsets (min).

    Returns:
        List of OnsetEvent records ordered by time.
    """
    n_time = images.shape[0]
    bg_steps = int(round(bg_window_min / cadence_min))

    onsets: List[OnsetEvent] = []
    last_accepted_time = -np.inf

    # Iterate forward in time, leaving room to look ahead for expansion.
    look_ahead_steps = int(round(min_expansion_min / cadence_min))
    for t in range(bg_steps, n_time - look_ahead_steps):
        background = images[t - bg_steps:t].mean(axis=0)
        bg_sigma = images[t - bg_steps:t].std(axis=0) + 1e-6
        threshold = background + n_sigma * bg_sigma

        bright_mask = images[t] > threshold
        if not bright_mask.any():
            continue

        # Criterion 1: local brightening; pick brightest pixel within mask.
        flat_index = int(np.argmax(images[t] * bright_mask))
        i_lat, i_mlt = np.unravel_index(flat_index, images[t].shape)
        mlat_peak = float(mlat_grid[i_lat])
        mlt_peak = float(mlt_grid[i_mlt])

        # Criterion 2: azimuthal expansion over the next min_expansion_min.
        future_mlts = []
        for dt in range(1, look_ahead_steps + 1):
            future_mask = images[t + dt] > threshold
            if future_mask.any():
                future_mlts.extend(mlt_grid[np.any(future_mask, axis=0)].tolist())
        if not future_mlts:
            continue
        mlt_extent = max(future_mlts) - min(future_mlts)
        if mlt_extent < min_expansion_mlt:
            continue

        # Criterion 3: 30-min separation from last accepted onset.
        time_min = t * cadence_min
        if time_min - last_accepted_time < min_separation_min:
            continue

        onsets.append(OnsetEvent(
            time_min=time_min,
            mlt_hr=mlt_peak,
            mlat_deg=mlat_peak,
            peak_count=float(images[t, i_lat, i_mlt]),
        ))
        last_accepted_time = time_min

    return onsets

### Synthetic FUV image stack / 합성 FUV 이미지 스택

**EN.** We synthesize a 24-hour run of FUV-like images at 2-min cadence, with three substorm onsets injected at known times, MLTs, and latitudes. Each onset starts as a localized Gaussian intensity bump, then expands in MLT over 25 minutes.

**KR.** 2분 cadence로 24시간 분량의 FUV 유사 이미지를 합성하고, 알려진 시각·MLT·위도에서 세 개의 substorm onset을 주입한다. 각 onset은 국지적 가우시안 전서에서 시작해 25분에 걸쳐 MLT 방향으로 확장한다.

In [ ]:
def synthesize_fuv_stack(duration_hr: float = 24.0,
                         cadence_min: float = 2.0,
                         injected_onsets: List[Tuple[float, float, float]] = None
                         ) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Generate a synthetic FUV image stack with injected substorm onsets.

    Args:
        duration_hr: Total simulated duration in hours.
        cadence_min: Cadence between frames (min).
        injected_onsets: List of (time_min, mlt_hr, mlat_deg) triples for true onsets.

    Returns:
        images: ndarray of shape (n_time, n_mlat, n_mlt).
        mlt_grid: 1-D MLT bin centers.
        mlat_grid: 1-D magnetic-latitude bin centers.
    """
    if injected_onsets is None:
        injected_onsets = [(180.0, 22.5, 66.0),
                           (540.0, 23.5, 65.0),
                           (1020.0, 22.0, 67.5)]
    mlt_grid = np.linspace(18.0, 30.0, 49)  # 18 MLT — 06 MLT next day, wrapped.
    mlat_grid = np.linspace(60.0, 75.0, 31)
    n_time = int(duration_hr * 60 / cadence_min)
    images = rng.normal(loc=20.0, scale=3.0, size=(n_time, mlat_grid.size, mlt_grid.size))
    images = np.clip(images, 0.0, None)

    for t0_min, mlt0, mlat0 in injected_onsets:
        t0 = int(t0_min / cadence_min)
        for k, dt_step in enumerate(range(0, 13)):
            t = t0 + dt_step
            if t >= n_time:
                break
            mlt_spread = 0.4 + 0.15 * k
            mlat_spread = 0.5
            amplitude = 120.0 * np.exp(-((dt_step - 4.0) / 6.0) ** 2)
            mlt_2d, mlat_2d = np.meshgrid(mlt_grid, mlat_grid)
            bump = amplitude * np.exp(
                -((mlt_2d - mlt0) / mlt_spread) ** 2
                - ((mlat_2d - mlat0) / mlat_spread) ** 2
            )
            images[t] += bump
    return images, mlt_grid, mlat_grid


true_onsets = [(180.0, 22.5, 66.0), (540.0, 23.5, 65.0), (1020.0, 22.0, 67.5)]
fuv_images, mlt_grid, mlat_grid = synthesize_fuv_stack(injected_onsets=true_onsets)
print(f"Image stack shape: {fuv_images.shape}")
print(f"MLT range: {mlt_grid[0]:.1f}-{mlt_grid[-1]:.1f} h, MLAT range: {mlat_grid[0]:.0f}-{mlat_grid[-1]:.0f} deg")

In [ ]:
detected = detect_onsets(fuv_images, mlt_grid, mlat_grid)
print(f"Detected {len(detected)} onsets out of {len(true_onsets)} injected:")
for ev in detected:
    mlt_display = ev.mlt_hr if ev.mlt_hr < 24 else ev.mlt_hr - 24
    print(f"  t={ev.time_min:7.1f} min   MLT={mlt_display:5.2f} h   MLAT={ev.mlat_deg:5.1f} deg   peak={ev.peak_count:5.1f}")

## Part 2: Statistical Reproduction of the Frey et al. (2004) Catalog / Frey et al. (2004) 카탈로그 통계 재현

**EN.** Below we generate a synthetic 2437-event catalog whose statistics match the paper's reported distributions (median MLT 23:00, std 1.35 h; median MLAT 66.4°, std 2.86°). The catalog is then plotted as histograms reproducing Figure 3 of Frey et al. (2004).

**KR.** 아래에서는 논문에 보고된 분포(중앙값 MLT 23:00, 표준편차 1.35시간; 중앙값 MLAT 66.4°, 표준편차 2.86°)에 맞는 합성 2437개 onset 카탈로그를 생성한다. 이 카탈로그를 히스토그램으로 그려 Frey et al. (2004)의 Figure 3을 재현한다.

In [ ]:
def synthesize_frey_catalog(n_events: int = 2437,
                            mlt_mean: float = 23.0,
                            mlt_std: float = 1.35,
                            mlat_mean: float = 66.4,
                            mlat_std: float = 2.86,
                            seed: int = 42) -> dict:
    """Generate a synthetic catalog matching the Frey et al. (2004) statistics.

    Args:
        n_events: Number of onsets to generate (Frey et al. 2004: 2437).
        mlt_mean: Mean of the MLT distribution in hours (paper: 23.0).
        mlt_std: Standard deviation of the MLT distribution in hours (paper: 1.35).
        mlat_mean: Mean of the AACGM-latitude distribution in deg (paper: 66.4).
        mlat_std: Standard deviation of the AACGM-latitude distribution in deg (paper: 2.86).
        seed: Random seed for reproducibility.

    Returns:
        Dictionary with keys 'mlt_hr', 'mlat_deg', 'imf_bz_nT', 'pdyn_nPa', 'season'.
    """
    local_rng = np.random.default_rng(seed)
    mlt = local_rng.normal(mlt_mean, mlt_std, n_events)
    mlt = np.mod(mlt, 24.0)
    mlat = local_rng.normal(mlat_mean, mlat_std, n_events)
    imf_bz = local_rng.normal(-0.5, 4.0, n_events)
    pdyn = np.clip(local_rng.lognormal(mean=np.log(2.0), sigma=0.5, size=n_events), 0.3, 30.0)
    season = local_rng.integers(1, 13, size=n_events)

    # Bake in the published trends.
    mlat = mlat + 0.25 * imf_bz - 1.5 * np.log10(pdyn)
    summer_mask = (season >= 5) & (season <= 8)
    winter_mask = (season <= 2) | (season == 12)
    mlt = mlt + np.where(summer_mask, -0.5, 0.0) + np.where(winter_mask, +0.3, 0.0)
    mlt = np.mod(mlt, 24.0)

    return {
        'mlt_hr': mlt,
        'mlat_deg': mlat,
        'imf_bz_nT': imf_bz,
        'pdyn_nPa': pdyn,
        'season': season,
    }

catalog = synthesize_frey_catalog()
print(f"N onsets:       {len(catalog['mlt_hr'])}")
print(f"MLT  median:    {np.median(catalog['mlt_hr']):.2f} h   (paper 23.00)")
print(f"MLT  mean:      {np.mean(catalog['mlt_hr']):.2f} h   (paper 22.50)")
print(f"MLT  std:       {np.std(catalog['mlt_hr']):.2f} h   (paper 1.35)")
print(f"MLAT median:    {np.median(catalog['mlat_deg']):.2f} deg (paper 66.40)")
print(f"MLAT mean:      {np.mean(catalog['mlat_deg']):.2f} deg (paper 66.10)")
print(f"MLAT std:       {np.std(catalog['mlat_deg']):.2f} deg (paper 2.86)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel A: MLT histogram (bottom panel of paper Figure 3).
axes[0].hist(catalog['mlt_hr'], bins=np.arange(15, 28, 0.5), color='steelblue', edgecolor='k')
axes[0].axvline(np.median(catalog['mlt_hr']), color='red', linestyle='--',
                label=f"median = {np.median(catalog['mlt_hr']):.2f} h")
axes[0].axvline(23.0, color='black', linestyle=':', label='paper median (23:00)')
axes[0].set_xlabel('MLT (hours)')
axes[0].set_ylabel('Substorms')
axes[0].set_title('Onset MLT histogram (cf. Frey et al. 2004 Fig. 3 bottom)')
axes[0].legend()
axes[0].set_xlim(15, 28)

# Panel B: Magnetic-latitude histogram (middle panel of paper Figure 3).
axes[1].hist(catalog['mlat_deg'], bins=np.arange(55, 76, 0.5), color='darkorange', edgecolor='k')
axes[1].axvline(np.median(catalog['mlat_deg']), color='red', linestyle='--',
                label=f"median = {np.median(catalog['mlat_deg']):.2f} deg")
axes[1].axvline(66.4, color='black', linestyle=':', label='paper median (66.4 deg)')
axes[1].set_xlabel('Magnetic latitude (deg)')
axes[1].set_ylabel('Substorms')
axes[1].set_title('Onset MLAT histogram (cf. Frey et al. 2004 Fig. 3 middle)')
axes[1].legend()
axes[1].set_xlim(55, 76)

plt.tight_layout()
plt.show()

## Part 3: IMF Bz and Dynamic-Pressure Dependences / IMF Bz 와 동압력 의존성

**EN.** Frey et al. (2004) report two key trends: (a) IMF Bz controls onset latitude (more southward Bz → lower latitude), and (b) higher dynamic pressure pushes onsets equatorward following $\Lambda \propto -1.5 \log_{10}(P_{dyn})$. Below we recover both relationships from the synthetic catalog by simple linear regression.

**KR.** Frey et al. (2004)는 두 가지 핵심 경향을 보고한다: (a) IMF Bz가 onset 위도를 제어(더 남향 Bz → 더 낮은 위도), (b) 동압력이 높을수록 onset이 적도쪽으로 이동 ($\Lambda \propto -1.5 \log_{10}(P_{dyn})$). 아래에서는 합성 카탈로그에서 선형회귀로 두 관계를 재구성한다.

In [ ]:
def linear_fit(x: np.ndarray, y: np.ndarray) -> Tuple[float, float]:
    """Return slope and intercept of a least-squares linear fit y = a x + b.

    Args:
        x: Independent variable.
        y: Dependent variable.

    Returns:
        Tuple of (slope, intercept).
    """
    a, b = np.polyfit(x, y, 1)
    return float(a), float(b)


fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# IMF Bz vs MLAT.
bz_bins = np.arange(-10, 11, 1)
bin_centers, mlat_means = [], []
for low in bz_bins[:-1]:
    sel = (catalog['imf_bz_nT'] >= low) & (catalog['imf_bz_nT'] < low + 1)
    if sel.sum() > 5:
        bin_centers.append(low + 0.5)
        mlat_means.append(catalog['mlat_deg'][sel].mean())
bin_centers, mlat_means = np.array(bin_centers), np.array(mlat_means)
slope, intercept = linear_fit(bin_centers, mlat_means)

axes[0].scatter(catalog['imf_bz_nT'], catalog['mlat_deg'], s=4, alpha=0.15, color='steelblue')
axes[0].plot(bin_centers, mlat_means, 'ro-', label='binned mean')
axes[0].plot(bin_centers, slope * bin_centers + intercept, 'k--',
             label=f'fit: MLAT = {slope:.2f} Bz + {intercept:.1f}')
axes[0].set_xlabel('IMF Bz (nT)')
axes[0].set_ylabel('Onset magnetic latitude (deg)')
axes[0].set_title('IMF Bz vs onset latitude')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Dynamic pressure vs MLAT.
log_p = np.log10(catalog['pdyn_nPa'])
p_bins = np.arange(-0.5, 1.5, 0.2)
p_centers, p_mlat_means = [], []
for low in p_bins[:-1]:
    sel = (log_p >= low) & (log_p < low + 0.2)
    if sel.sum() > 5:
        p_centers.append(low + 0.1)
        p_mlat_means.append(catalog['mlat_deg'][sel].mean())
p_centers, p_mlat_means = np.array(p_centers), np.array(p_mlat_means)
p_slope, p_intercept = linear_fit(p_centers, p_mlat_means)

axes[1].scatter(log_p, catalog['mlat_deg'], s=4, alpha=0.15, color='darkorange')
axes[1].plot(p_centers, p_mlat_means, 'ro-', label='binned mean')
axes[1].plot(p_centers, p_slope * p_centers + p_intercept, 'k--',
             label=f'fit: MLAT = {p_slope:.2f} log10(Pdyn) + {p_intercept:.1f}')
axes[1].set_xlabel('log10 Pdyn (nPa)')
axes[1].set_ylabel('Onset magnetic latitude (deg)')
axes[1].set_title('Dynamic pressure vs onset latitude')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"IMF Bz slope:                {slope:+.3f} deg/nT  (paper: ~ +0.2 to +0.3 deg/nT)")
print(f"log10(Pdyn) slope:           {p_slope:+.2f} deg/dex (paper: -1.5 deg/dex)")

## Part 4: Seasonal MLT Shift / 계절성 MLT 이동

**EN.** Liou et al. (2001) and Frey et al. (2004) both report that summer onsets occur about 1 hour earlier in MLT than winter onsets. We verify this on the synthetic catalog.

**KR.** Liou et al. (2001) 및 Frey et al. (2004)는 여름 onset이 겨울보다 MLT 기준 약 1시간 이르다고 보고한다. 합성 카탈로그에서 이를 확인한다.

In [ ]:
season_labels = ['DJF', 'MAM', 'JJA', 'SON']
season_groups = [(12, 1, 2), (3, 4, 5), (6, 7, 8), (9, 10, 11)]

fig, ax = plt.subplots(figsize=(10, 5))
season_medians = []
for label, months in zip(season_labels, season_groups):
    sel = np.isin(catalog['season'], months)
    if sel.sum() == 0:
        continue
    ax.hist(catalog['mlt_hr'][sel], bins=np.arange(15, 28, 0.5), alpha=0.5, label=f'{label} (n={sel.sum()})')
    season_medians.append((label, np.median(catalog['mlt_hr'][sel])))

ax.set_xlabel('MLT (hours)')
ax.set_ylabel('Substorms')
ax.set_title('Onset MLT by season (cf. Liou et al. 2001 ; Frey et al. 2004)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

for label, med in season_medians:
    print(f"{label} median MLT: {med:.2f} h")
summer_med = next(m for lbl, m in season_medians if lbl == 'JJA')
winter_med = next(m for lbl, m in season_medians if lbl == 'DJF')
print(f"\nSummer-winter MLT difference: {summer_med - winter_med:+.2f} h (paper: ~ -1 h)")

## Summary / 요약

| Concept / 개념 | This Paper / 이 논문 | Modern Equivalent / 현대적 동등물 |
|---|---|---|
| Onset detection rule / onset 검출 규칙 | Visual + brightest-pixel + 20-min/30-min rule | SuperMAG SML index trigger; CNN-based onset detectors (Maimaiti 2019) |
| Onset catalog / onset 카탈로그 | 2437 events, public ASCII list | SuperMAG substorm list (Newell & Gjerloev 2011); Ohtani-Gjerloev list |
| MLT distribution / MLT 분포 | Median 23:00, std 01:21 | Reproduced by THEMIS-ASI, SuperMAG, OMNI-driven simulations |
| Latitude distribution / 위도 분포 | 66.4° ± 2.86° (AACGM) | Same to within 0.5° in all subsequent statistical surveys |
| IMF Bz dependency / Bz 의존 | Southward Bz → lower latitude | Confirmed via SuperDARN, OMNI; quantified in Newell-Gjerloev SML |
| Dynamic-pressure dependency / 동압력 의존 | $-1.5\,\log_{10}(P_{dyn})$ | Same trend in GEOTAIL/THEMIS plasma-sheet inner-edge studies |

**EN.** The paper's three-rule onset definition has become a de facto standard, and the published 2437-event list still serves as ground truth for modern statistical and machine-learning substorm studies.

**KR.** 본 논문의 3-rule onset 정의는 사실상 표준이 되었고, 공개된 2437개 onset 리스트는 현대 통계·기계학습 substorm 연구의 ground truth로 계속 사용되고 있다.